# Chat with a Self-Hosted LLM Using Microsoft Agent Framework

This notebook demonstrates how to use the **Microsoft Agent Framework** to build an AI agent that chats with a self-hosted Gemma 4 model running on Azure Container Apps (ACA) with vLLM.

The Agent Framework's `OpenAIChatClient` connects to any OpenAI-compatible endpoint, including the vLLM server deployed in this project.

In [1]:
%pip install agent-framework==1.0.1

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Get the LLM Endpoint

Retrieve the FQDN of the Gemma 4 model deployed on ACA from the Terraform output.

In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.icyocean-d166d91d.swedencentral.azurecontainerapps.io


## 1. Simple Agent — Single Turn

Create an agent backed by the self-hosted Gemma 4 model via `OpenAIChatClient` pointing at the vLLM OpenAI-compatible endpoint.

In [3]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
)

async with Agent(
    client=client,
    instructions="You are a helpful AI assistant. Be concise and informative.",
) as agent:
    result = await agent.run("What is Azure Container Apps?")
    print(result.text)

**Azure Container Apps (ACA)** is a fully managed, serverless container service designed for building and deploying modern, microservices-based applications. 

It abstracts away the complexity of managing underlying infrastructure (like Kubernetes), allowing developers to focus on writing code rather than configuring clusters.

### Key Features
*   **Serverless Scaling:** It automatically scales your containers based on HTTP traffic or events (KEDA). It can even **scale to zero** when not in use to save costs.
*   **Built on Open Source:** It is powered by **Kubernetes**, **KEDA** (event-driven autoscaling), **Dapr** (distributed application runtime), and **Envoy** (proxy).
*   **Simplified Networking:** It includes built-in ingress (HTTP/TCP) and traffic splitting, making it easy to perform **Blue-Green** or **Canary** deployments.
*   **Microservices Support:** Through the integration of **Dapr**, it simplifies common microservice challenges like state management, service-to-service 

## 2. Streaming Response

Use `stream=True` for a token-by-token streaming experience, which is the recommended pattern for production-grade apps.

In [4]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
)

async with Agent(
    client=client,
    instructions="You are a helpful AI assistant. Be concise and informative.",
) as agent:
    print("Agent: ", end="", flush=True)

    stream = agent.run("Explain Kubernetes in 3 sentences.", stream=True)
    
    async for chunk in stream:
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print()
    await stream.get_final_response()  # finalize the stream

Agent: Kubernetes is an open-source orchestration platform designed to automate the deployment, scaling, and management of containerized applications. It ensures high availability by monitoring containers and automatically restarting or replacing those that fail. By distributing workloads across a cluster of machines, it optimizes resource usage and allows applications to scale seamlessly based on demand.


## 3. Agent with Tool Calling

Enhance the agent with custom Python functions as tools. The Agent Framework automatically handles the tool-calling loop with the LLM.

In [7]:
from typing import Annotated
from random import randint
from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient


# NOTE: approval_mode="never_require" is for sample brevity.
# Use "always_require" in production for user confirmation before tool execution.
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."

# def get_time(
#     timezone: Annotated[str, "The timezone, e.g. 'UTC', 'CET', 'PST'."],
# ) -> str:
#     """Get the current time in a given timezone."""
#     from datetime import datetime
#     return f"The current time in {timezone} is {datetime.now().strftime('%H:%M:%S')} (simulated)."


agent = Agent(
    client=OpenAIChatClient(
        base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
        api_key="EMPTY",
        model="google/gemma-4-31B-it",
    ),
    name="WeatherAgent",
    instructions="You are a helpful assistant that can provide weather and restaurant information.",
    tools=[get_weather],
)

result = await agent.run("What's the weather like in Seattle?")
print(f"Agent: {result}")

# print("Agent: ", end="", flush=True)

# stream = await agent.run("What's the weather in Amsterdam and what are today's specials?", stream=True)
# async for chunk in stream:
#     if chunk.text:
#         print(chunk.text, end="", flush=True)
# print()
# await stream.get_final_response()
       

ChatClientException: <class 'agent_framework_openai._chat_client.OpenAIChatClient'> service failed to complete the prompt: Error code: 404 - {'error': {'message': "Response with id 'resp_9adaa5674a9a6bb9' not found.", 'type': 'invalid_request_error', 'param': 'response_id', 'code': 404}}

In [2]:
from agent_framework import Agent, ChatOptions, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatClient

# client = OpenAIChatClient(
#     base_url=f"http://127.0.0.1:11434/v1/", 
#     model="gemma4:31b",
#     api_key="EMPTY"
# )

client = OpenAIChatClient(
    base_url="https://foundry-555.openai.azure.com/openai/v1",
    model="gpt-5.2",
    api_key="7lDCTqHKRZDATNxBKqZk3QKh8EoFlRVvMIYGeJds0fWad81JYiLAJQQJ99CDACfhMk5XJ3w3AAAAACOGlf3w"
)

# client = OpenAIChatClient(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     model="google/gemma-4-31B-it",
#     api_key="EMPTY"
# )

async with (
    MCPStreamableHTTPTool(
        name="Microsoft Learn MCP",
        url="https://learn.microsoft.com/api/mcp",
    ) as mcp_server,
    Agent(
        client=client,
        name="DocsAgent",
        instructions="You help with Microsoft documentation questions.",
    ) as agent,
):
    response = await agent.run(
        "How to create an Azure storage account using az cli?",
        tools=mcp_server,
        options=ChatOptions(max_tokens=1024, temperature=0.9),
    )

    print(f"Agent: {response.text}")

Agent: Use `az storage account create`. A minimal end-to-end example:

```bash
# sign in (skip if already signed in, e.g., in Cloud Shell)
az login

# create a resource group
az group create \
  --name storage-resource-group \
  --location eastus

# create a StorageV2 (general-purpose v2) storage account
# (account name must be globally unique, 3-24 lowercase letters/numbers)
az storage account create \
  --name <account-name> \
  --resource-group storage-resource-group \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

Verify:

```bash
az storage account show -g storage-resource-group -n <account-name>
```

Docs: https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account and CLI reference: https://learn.microsoft.com/cli/azure/storage/account?view=azure-cli-latest#az-storage-account-create
